Evaluation

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [2]:
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama
from langchain_classic.document_loaders import CSVLoader
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [3]:
### 构造一个LLM文档问答应用，如"04_QnA"

file = "data/OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file, encoding="UTF-8")
data = loader.load()

llm = ChatOllama(model=os.getenv('MODEL_NAME'), temperature=0)

# 指定向量存储类
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,  # 指定向量存储类
    embedding=HuggingFaceEmbeddings()
).from_loaders([loader])

# 船舰检索QA链
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=index.vectorstore.as_retriever(),
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
### 测试数据
data[10]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.")

In [5]:
data[11]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

hard-coded examples

In [6]:
### hard-coded examples：手动创建测试数据，每个“问答对”包含一个query和一个answer
### 手动构造多个问答对，让LLM给出回答并与准备好的答案作比较，最后给LLM打分。（耗时耗力）

examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes."
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection."
    }
]

通过LLM生成测试用例

In [7]:
### LLM-Generated examples
from langchain_classic.evaluation.qa import QAGenerateChain

example_gen_chain = QAGenerateChain.from_llm(llm=llm)

new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

C:\Users\61075\AppData\Local\Temp\ipykernel_33948\163079534.py:6: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  new_examples = example_gen_chain.apply_and_parse(


In [8]:
#查看用例数据
print("新生成的问答用例：\n", new_examples[0])

print("\n原始数据：\n", data[0])

新生成的问答用例：
 {'qa_pairs': {'query': "What is the approximate weight of a pair of Women's Campside Oxfords?", 'answer': "The approximate weight of a pair of Women's Campside Oxfords is 1 lb. 1 oz."}}

原始数据：
 page_content=': 0
name: Women's Campside Oxfords
description: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. 

Size & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. 

Specs: Approx. weight: 1 lb.1 oz. per pair. 

Construction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. 

Questions? Please contact us for any inquiries.' metadat

In [9]:
# combine examples：合测试集
examples += new_examples

# 手写的example中key为query和answer；后面5个是QAGenerateChain生成的，query和answer被包裹在qa_pairs里面
# 对齐
cleaned_examples = []
for item in examples:
    # 处理 QAGenerateChain 生成的嵌套格式
    if "qa_pairs" in item:
        cleaned_examples.append(item["qa_pairs"])
    else:
        cleaned_examples.append(item)
print(cleaned_examples)

[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?', 'answer': 'Yes.'}, {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?', 'answer': 'The DownTek collection.'}, {'query': "What is the approximate weight of a pair of Women's Campside Oxfords?", 'answer': "The approximate weight of a pair of Women's Campside Oxfords is 1 lb. 1 oz."}, {'query': 'What are the dimensions of the Small Recycled Waterhog dog mat, and what material is it made from?', 'answer': 'The dimensions of the Small Recycled Waterhog dog mat are 18" x 28", and it is made from 94% recycled materials.'}, {'query': "What features does the Infant and Toddler Girls' Coastal Chill Swimsuit have to ensure comfort and protection?", 'answer': "The swimsuit ensures comfort and protection through its four-way-stretch and chlorine-resistant fabric, which keeps its shape and resists snags. It also includes UPF 50+ rated fabric that provides high sun protection by blocking 98% of the sun's

In [10]:
qa.invoke(input=cleaned_examples[3]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'What are the dimensions of the Small Recycled Waterhog dog mat, and what material is it made from?',
 'result': 'The dimensions of the Small Recycled Waterhog dog mat are 18" x 28". It is made from 24 oz. polyester fabric that is 94% recycled materials.'}

Manual evaluation (and debugging)

In [16]:
# 全局开启所有链条的详细执行日志
from langchain_core.globals import set_debug
set_debug(True)

# 重新运行与上面相同的示例，可以看到它开始打印出更多的信息
qa.invoke(input=cleaned_examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set have side pockets?",
  "context": ": 73\nname: Cozy Cuddles Knit Pullover Set\ndescription: Perfect for lounging, this knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out. \n\nSize & Fit \nPants are Favorite Fit: Sits lower on the waist. \nRelaxed Fit: Our most generous fit sits farthest from the body. \n\nFabric & Care \nIn the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features \nRelaxed fit top with raglan sleeves and rounded hem. \nPull-on pants have a wide elastic

{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
 'result': 'Yes, the Cozy Comfort Pullover Set has side pockets in the pull-on pants.'}

In [17]:
# Turn off the debug mode
set_debug(False)

Overall：LLM assisted evaluation
- 首先，使用LLM自动创建一个问答测试集，包含query和answer；
- 然后，使用LLM回答测试机中问题，得到responses；
- 接下来，评估回答是否正确

In [11]:
#为所有不同的示例创建预测
predictions = qa.apply(cleaned_examples)

C:\Users\61075\AppData\Local\Temp\ipykernel_33948\1540675912.py:2: LangChainDeprecationWarning: The method `Chain.apply` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `batch` instead.
  predictions = qa.apply(cleaned_examples)




> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


In [14]:
# 对预测的结果进行评估，导入QA问题回答，评估链，通过语言模型创建此链
from langchain_classic.evaluation.qa import QAEvalChain     #导入QA问题回答评估链

llm = ChatOllama(model=os.getenv("MODEL_NAME"), temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

#在此链上调用evaluate，进行评估
graded_outputs = eval_chain.evaluate(cleaned_examples, predictions)

In [20]:
for i, eg in enumerate(cleaned_examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['results'])
    print()

Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real Answer: Yes.
Predicted Answer: Yes, the Cozy Comfort Pullover Set has pull-on pants with side pockets.
Predicted Grade: GRADE: CORRECT

The student's answer is factually accurate. While it provides additional information about the pull-on pants, this does not conflict with the true answer and correctly states that the Cozy Comfort Pullover Set has side pockets.

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection.
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Predicted Grade: CORRECT

The student's answer matches the true answer in terms of factual accuracy, even though it includes a slight additional detail ("Ultra-Lofty 850 Stretch Down Hooded Jacket") that does not conflict with the true answer.

Example 2:
Question: What is the approximate weight of a pair of Women's Campside Oxf